# Error Handling in Practice

## What you'll learn

- How the framework catches and records runtime failures
- Finding and reading failure logs
- Partial failures with `FailurePolicy` (CONTINUE vs FAIL_FAST)
- Empty input cascades from strict filters
- Validation errors vs runtime errors

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Exploring Results](../01-getting-started/02-exploring-results.ipynb)  
**Estimated time:** 15 minutes  
**GPU required:** No.

---

Inspect the outcomes of invalid data, partial failures, filters that reject
every artifact, and invalid configuration.

In [ ]:
from __future__ import annotations

from artisan.operations.curator import Filter, IngestCSV
from artisan.operations.examples import DataGeneratorWithMetrics, DataTransformer
from artisan.orchestration import PipelineManager, StepStatus
from artisan.schemas import FailurePolicy
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_failures, inspect_pipeline

## A step that fails

Create a CSV with `"INVALID"` in a numeric column, ingest it, then pass it to
`DataTransformer`. `IngestCSV` reads the CSV headers and row count without
validating numeric values. The transformer fails when it tries `float("INVALID")`.

In [ ]:
import os

env = tutorial_setup("error_handling")

# Create a CSV with a non-numeric value in the 'score' column
bad_csv = os.path.join(env.working_root, "bad_data.csv")
with open(bad_csv, "w") as f:
    f.write("id,x,y,z,score\n0,1.0,2.0,3.0,INVALID\n")
print(f"Bad CSV: {bad_csv}")
with open(bad_csv) as f:
    print(f.read())

In [ ]:
pipeline = PipelineManager.create(
    name="error_handling",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

# Step 0: Ingest the CSV (succeeds without validating numeric values)
step_0 = pipeline.run(
    operation=IngestCSV,
    name="ingest",
    inputs=[bad_csv],
)

# Step 1: Transform — fails when parsing the non-numeric value
step_1 = pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("ingest", "csv")},
)

result = pipeline.finalize()
assert step_0.status is StepStatus.SUCCEEDED
assert step_1.status is StepStatus.FAILED
print(f"Pipeline complete: success={result['overall_success']}")

In [ ]:
print(
    f"Step 0 (ingest):    status={step_0.status.value}, succeeded={step_0.succeeded_count}, failed={step_0.failed_count}"
)
print(
    f"Step 1 (transform): status={step_1.status.value}, succeeded={step_1.succeeded_count}, failed={step_1.failed_count}"
)

In [ ]:
inspect_pipeline(env.delta_root)

`inspect_pipeline` shows `"failed"` because no transform execution succeeded.
Under `CONTINUE`, a mixture of successes and failures would be `"partial"`.

Use `inspect_failures` below to read execution errors, including `code`,
`recovery_hint`, and `field` when the failure is structured.

## Finding and reading failure logs

When an execution fails, the framework writes a human-readable log
file at `<runs_dir>/logs/failures/YYYYMMDD/YYYYMMDDTHHMMSSffffffZ_executionID.log`. Each
file contains the run ID, operation name, a traceback, and the
last 100 lines of tool output, with recognized sensitive values redacted.
See [Debug Executions](../../how-to-guides/debugging-executions.md) for replay
configuration and secret handling.

In [ ]:
import os

logs_dir = os.path.join(env.runs_dir, "logs", "failures")
if os.path.exists(logs_dir):
    for step_name in sorted(os.listdir(logs_dir)):
        step_dir = os.path.join(logs_dir, step_name)
        if not os.path.isdir(step_dir):
            continue
        for log_name in sorted(os.listdir(step_dir)):
            log_file = os.path.join(step_dir, log_name)
            rel_path = os.path.relpath(log_file, env.runs_dir)
            print(f"--- {rel_path} ---")
            with open(log_file) as f:
                print(f.read())
else:
    print("No failure logs found")

The log shows the exact error (`ValueError: could not convert string
to float: 'INVALID'`) and the traceback pointing to
`data_transformer.py`. The **execution run ID** in the log matches the
`execution_run_id` column in the executions Delta table, so you can
cross-reference failures programmatically.

In [ ]:
inspect_failures(env.delta_root)

## Partial failures with FailurePolicy

`FailurePolicy` determines what happens when some inputs fail and others succeed:

| Policy | Behavior |
|--------|----------|
| `CONTINUE` (default) | Commit successful results and return a `partial` StepResult |
| `FAIL_FAST` | Return a `failed` StepResult; already-finished records remain available for audit |

We'll create 5 CSVs where 2 have invalid data, then run with each
policy.

In [ ]:
import os

env_partial = tutorial_setup("error_handling_partial")

# Create 5 CSVs — indices 1 and 3 have non-numeric values
csv_dir = os.path.join(env_partial.working_root, "csvs")
os.makedirs(csv_dir, exist_ok=True)

csv_paths = []
for i in range(5):
    path = os.path.join(csv_dir, f"data_{i:03d}.csv")
    if i in (1, 3):
        content = f"id,x,y,z,score\n{i},1.0,2.0,3.0,INVALID\n"
    else:
        content = f"id,x,y,z,score\n{i},1.0,2.0,3.0,0.5\n"
    with open(path, "w") as f:
        f.write(content)
    csv_paths.append(path)

print(f"Created {len(csv_paths)} CSVs (indices 1, 3 have bad data)")

In [ ]:
pipeline = PipelineManager.create(
    name="partial_continue",
    delta_root=env_partial.delta_root,
    staging_root=env_partial.staging_root,
    working_root=env_partial.working_root,
)
output = pipeline.output

step_0 = pipeline.run(
    operation=IngestCSV,
    name="ingest",
    inputs=csv_paths,
)

# Default policy is CONTINUE — failures are logged, successes committed
step_1 = pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("ingest", "csv")},
)

result = pipeline.finalize()
assert step_1.status is StepStatus.PARTIAL
assert step_1.succeeded_count == 3
assert step_1.failed_count == 2

print(f"Step 1: succeeded={step_1.succeeded_count}, failed={step_1.failed_count}")
print(f"Pipeline success={result['overall_success']}")

In [ ]:
inspect_pipeline(env_partial.delta_root)

With `CONTINUE`, the step became `partial` with 3 successes and 2 failures.
The 3 successful outputs were committed to Delta Lake. Any downstream
step would receive only those 3 artifacts.

Run the same pipeline with `FAIL_FAST`:

In [ ]:
env_fast = tutorial_setup("error_handling_failfast")

pipeline = PipelineManager.create(
    name="partial_failfast",
    delta_root=env_fast.delta_root,
    staging_root=env_fast.staging_root,
    working_root=env_fast.working_root,
    failure_policy=FailurePolicy.FAIL_FAST,
)
output = pipeline.output

step_0 = pipeline.run(
    operation=IngestCSV,
    name="ingest",
    inputs=[str(p) for p in csv_paths],
)

step_1 = pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("ingest", "csv")},
)
assert step_1.status is StepStatus.FAILED
print(f"FAIL_FAST step status: {step_1.status.value}")
pipeline.finalize()

With `FAIL_FAST`, the step is `failed` and exposes no output roles to
downstream steps. Failure records and sibling work that already finished
remain persisted for audit. Use `FAIL_FAST` when partial output is not meaningful.

## Empty input cascades

When a filter rejects every artifact, the pipeline skips downstream work and
records why. Generate `mean_score` values in [0, 1], then filter with the
impossible threshold > 100.

In [ ]:
env_cascade = tutorial_setup("error_handling_cascade")

pipeline = PipelineManager.create(
    name="empty_cascade",
    delta_root=env_cascade.delta_root,
    staging_root=env_cascade.staging_root,
    working_root=env_cascade.working_root,
)
output = pipeline.output

step_0 = pipeline.run(
    operation=DataGeneratorWithMetrics,
    name="generate",
    params={"count": 5, "seed": 42},
)

step_1 = pipeline.run(
    operation=Filter,
    name="filter",
    inputs={"passthrough": output("generate", "datasets")},
    params={
        "criteria": [{"metric": "mean_score", "operator": "gt", "value": 100}],
    },
)

step_2 = pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("filter", "passthrough")},
)

result = pipeline.finalize()
assert step_2.status is StepStatus.SKIPPED
assert step_2.metadata["skip_reason"] == "empty_inputs"
print(f"Pipeline complete: success={result['overall_success']}")

In [ ]:
inspect_pipeline(env_cascade.delta_root)

In [ ]:
for step in [step_0, step_1, step_2]:
    if step.status is StepStatus.SKIPPED:
        reason = step.metadata["skip_reason"]
        print(f"Step {step.step_number} ({step.step_name}): skipped — {reason}")
    else:
        print(
            f"Step {step.step_number} ({step.step_name}): {step.succeeded_count} succeeded"
        )

The filter passed zero artifacts, so step 2 was skipped with
`skip_reason="empty_inputs"`. If there were more downstream steps,
they would show `"pipeline_stopped"`.

:::{note}
To evaluate filter criteria without removing artifacts, use
`passthrough_failures=True` in the Filter params. This records
which artifacts would fail but passes everything through.
:::

## Validation before execution

Unknown parameter names and invalid input roles raise at the public call
boundary, before a step attempt is created. A bad value for a known parameter
is validated while preparing an allocated attempt. `pipeline.run()` returns
a failed `StepResult` and dispatches no execution work.

Use nonempty inputs in this example: an empty-input step can skip before
parameter-value validation. To validate values before creating a pipeline,
construct the operation's `Params` model directly.

In [ ]:
import pydantic

# Explicit parameter validation raises before any pipeline is created.
validation_errors = []
try:
    DataTransformer.Params(scale_factor="not_a_number")
except pydantic.ValidationError as exc:
    validation_errors = exc.errors()
    print(f"{type(exc).__name__}: {exc}")

assert validation_errors, "The invalid scale factor must be rejected"
assert validation_errors[0]["loc"] == ("scale_factor",)

env_val = tutorial_setup("error_handling_validation")
pipeline = PipelineManager.create(
    name="validation_demo",
    delta_root=env_val.delta_root,
    staging_root=env_val.staging_root,
    working_root=env_val.working_root,
)
source = pipeline.run(
    DataGeneratorWithMetrics,
    params={"count": 1, "seed": 42},
)
invalid = pipeline.run(
    DataTransformer,
    name="transform",
    inputs={"dataset": source.output("datasets")},
    params={"scale_factor": "not_a_number"},
)
validation_summary = pipeline.finalize()
assert invalid.status is StepStatus.FAILED
assert "scale_factor" in invalid.error
assert invalid.total_count == 0
assert not invalid.output_roles
assert not validation_summary["overall_success"]
print(invalid.error)
inspect_pipeline(env_val.delta_root)

The explicit `Params` check raised `ValidationError`. The pipeline example
instead persisted a failed transform attempt with zero processed items and
no accepted outputs. Its diagnostic is available as `invalid.error`.

This preparation failure has no worker execution or execution failure log.
The bad-CSV example above did reach a worker, so its error is also available
through `inspect_failures` and a failure log.

## Next steps

- [Storage Layout and Logging](../06-storage/01-storage-and-logging.ipynb) —
  Directory layout, Delta Lake tables, and logging configuration
- [Error Handling](../../concepts/error-handling.md) — Design
  rationale for failure containment
- [Resume and Caching](../03-caching/01-resume-and-caching.ipynb) — How caching
  interacts with failures during re-runs